# 04 · 数据与训练消融（推荐扩展 2）

**维度**：数据规模(500/2k/全量) × 数据质量(噪声0/0.2/0.5) × LoRA rank(4/8/32)
**分析**：成功/失败样例归类

In [ ]:
# [0] GPU 自检：确认拿到 T4，并钉死 T4 兼容参数
import torch
assert torch.cuda.is_available(), "未分配到 GPU！请 运行时->更改运行时类型 选 T4 GPU"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {name} | VRAM: {vram:.1f} GB | CC: {torch.cuda.get_device_capability(0)}")
# T4 = sm7.5：无原生 bf16、无 FlashAttention-2 -> 统一 fp16 + sdpa
DTYPE, ATTN = "float16", "sdpa"
print("钉死参数: dtype=%s attn=%s (T4 无 bf16/FA2)" % (DTYPE, ATTN))


In [ ]:
# [1] 安装依赖（T4 / CUDA12 兼容钉死版本）
# 关键：transformers 必须 <5（trl 0.x 与 5.x 不兼容）；不装 flash-attn（T4 sm7.5 不支持）
!pip install -q "torch==2.3.1" "transformers==4.44.2" "datasets==2.20.0" "peft==0.12.0" \
    "trl==0.9.6" "accelerate==0.33.0" "bitsandbytes==0.43.3" "sentencepiece" "tiktoken" \
    "numpy<2" "pandas" "matplotlib"
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)


In [ ]:
# [2] 克隆仓库并安装 minillm 包（单一事实源）
import os, subprocess
if not os.path.exists("cs0502-mini-llm"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Zhe-SH-CN/cs0502-mini-llm.git"], check=True)
os.chdir("cs0502-mini-llm")
!pip install -q -e .
import minillm
from minillm.config import ExpConfig
from minillm import data as D
from minillm import evaluate as E
from minillm import train as T
from minillm import quantize as Q
from minillm import bench as B
from minillm.record import Recorder
print("minillm", minillm.__version__, "| cwd", os.getcwd())


In [ ]:
# [3] 挂载 Google Drive，统一结果/数据落盘路径（Colab 文件系统易失，必须落 Drive）
from google.colab import drive
drive.mount("/content/drive")
import os
BASE = "/content/drive/MyDrive/cs0502-mini-llm"
DATA = os.path.join(BASE, "data"); RESULTS = os.path.join(BASE, "results")
os.makedirs(DATA, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)
rec = Recorder(RESULTS)
print("DATA:", DATA, "| RESULTS:", RESULTS)


In [ ]:
# 结果打表辅助
import pandas as pd
def show(df, title=""):
    if title: print("\n==", title, "==")
    from IPython.display import display
    display(df)
    return df


In [ ]:
import torch as _t
def _ft(d): return {'float16':_t.float16,'bfloat16':_t.bfloat16,'float32':_t.float32}[d]
import pandas as pd

In [ ]:
# 消融 A：数据规模
rows = []
for n in (200, 800, None):
    cfg = ExpConfig(run_id=f"abl-size-{n}", model_key="minimind-26m", task="courseqa", method="sft",
                    max_train_samples=n, dtype=DTYPE, attn=ATTN, epochs=2)
    out = os.path.join(RESULTS, "ckpt", cfg.run_id)
    T.train(D.load_dataset("courseqa", DATA), "courseqa", cfg, out)
    base, tok = E.load_model(ExpConfig(run_id="t", model_key="minimind-26m", dtype=DTYPE, attn=ATTN))
    from transformers import AutoModelForCausalLM
    base = AutoModelForCausalLM.from_pretrained(os.path.join(out,"final"), torch_dtype=_ft(DTYPE), attn_implementation=ATTN).eval()
    m, ps = E.evaluate(base, tok, D.load_dataset("courseqa", DATA), "courseqa", cfg)
    rec.log_run(cfg.run_id, cfg.to_dict(), m, ps)
    rows.append({"n_train": n, **{k: round(v,4) for k,v in m.items() if k in ("exact_match","token_f1","overall")}})
    del base; _t.cuda.empty_cache()
show(pd.DataFrame(rows), "ABLATION: DATA SIZE")


In [ ]:
# 消融 B：数据质量（标签噪声）
rows = []
for noise in (0.0, 0.2, 0.5):
    cfg = ExpConfig(run_id=f"abl-noise-{noise}", model_key="minimind-26m", task="intent", method="sft",
                    noise_ratio=noise, dtype=DTYPE, attn=ATTN, epochs=2)
    out = os.path.join(RESULTS, "ckpt", cfg.run_id)
    T.train(D.load_dataset("intent", DATA), "intent", cfg, out)
    from transformers import AutoModelForCausalLM
    base, tok = E.load_model(ExpConfig(run_id="t", model_key="minimind-26m", dtype=DTYPE, attn=ATTN))
    base = AutoModelForCausalLM.from_pretrained(os.path.join(out,"final"), torch_dtype=_ft(DTYPE), attn_implementation=ATTN).eval()
    m, ps = E.evaluate(base, tok, D.load_dataset("intent", DATA), "intent", cfg)
    rec.log_run(cfg.run_id, cfg.to_dict(), m, ps)
    rows.append({"noise": noise, **{k: round(v,4) for k,v in m.items() if k in ("accuracy","macro_f1")}})
    del base; _t.cuda.empty_cache()
show(pd.DataFrame(rows), "ABLATION: DATA QUALITY")


In [ ]:
# 消融 C：LoRA rank
rows = []
for r in (4, 8, 32):
    cfg = ExpConfig(run_id=f"abl-lora-r{r}", model_key="qwen2.5-0.5b", task="courseqa", method="lora",
                    lora_r=r, dtype=DTYPE, attn=ATTN, epochs=2)
    out = os.path.join(RESULTS, "ckpt", cfg.run_id)
    T.train(D.load_dataset("courseqa", DATA), "courseqa", cfg, out)
    from peft import PeftModel
    base, tok = E.load_model(ExpConfig(run_id="t", model_key="qwen2.5-0.5b", dtype=DTYPE, attn=ATTN))
    base = PeftModel.from_pretrained(base, os.path.join(out,"final")).merge_and_unload().eval()
    m, ps = E.evaluate(base, tok, D.load_dataset("courseqa", DATA), "courseqa", cfg)
    rec.log_run(cfg.run_id, cfg.to_dict(), m, ps)
    rows.append({"lora_r": r, **{k: round(v,4) for k,v in m.items() if k in ("exact_match","token_f1","overall")}})
    del base; _t.cuda.empty_cache()
show(pd.DataFrame(rows), "ABLATION: LORA RANK")


In [ ]:
# 成功/失败样例分析（取最近一次 courseqa run）
runs = [r for r in rec.load_runs() if r["config"]["task"]=="courseqa" and r["config"]["method"]=="sft"]
if runs:
    ps = runs[-1]["per_sample"]
    bad = [p for p in ps if p["score"] < 0.5][:8]
    good = [p for p in ps if p["score"] >= 0.9][:4]
    print("== 失败样例 ==")
    for p in bad: print(f'  Q: {p["q"]}\n  gold: {p["gold"]}\n  pred: {p["pred"][:80]}\n')
    print("== 成功样例 ==")
    for p in good: print(f'  Q: {p["q"]}\n  pred: {p["pred"][:80]}\n')
